In [1]:
!pip install -q skyfield gradio pandas

from datetime import datetime, timedelta, timezone
import gradio as gr
import pandas as pd
from skyfield.api import EarthSatellite, load

TARGET_SLOTS = ["Intelsat", "Eutelsat", "Starlink", "Qianfan", "Kuiper"]


def extract_single_tle(text, default_label):
  lines = [ln.strip() for ln in (text or "").strip().splitlines() if ln.strip()]
  i = 0
  while i + 1 < len(lines):
    if (
        not lines[i].startswith("1 ")
        and lines[i + 1].startswith("1 ")
        and (i + 2) < len(lines)
        and lines[i + 2].startswith("2 ")
    ):
      name, l1, l2 = lines[i], lines[i + 1], lines[i + 2]
      return f"{default_label} — {name}", l1, l2
    elif lines[i].startswith("1 ") and (i + 1) < len(lines) and lines[i + 1].startswith("2 "):
      l1, l2 = lines[i], lines[i + 1]
      return f"{default_label} ({l1[2:7].strip()})", l1, l2
    i += 1
  return None


def validate_slots(tle1, tle2, tle3, tle4, tle5):
  raw_list = [tle1, tle2, tle3, tle4, tle5]
  parsed = []
  missing = []
  for label, raw in zip(TARGET_SLOTS, raw_list):
    item = extract_single_tle(raw, label)
    if item is None:
      missing.append(label)
    else:
      parsed.append(item)
  return parsed, missing


def update_status_badge(tle1, tle2, tle3, tle4, tle5):
  parsed, missing = validate_slots(tle1, tle2, tle3, tle4, tle5)
  filled = len(parsed)
  if filled == 5:
    return f"""
        <div style="width:100%;box-sizing:border-box;padding:12px 16px;border-radius:10px;background:#0d281e;color:#34d399;border:1px solid #059669;font-family:Inter,sans-serif;font-weight:700;">
          ✅ 5 / 5 TLE slots ready — Click "Compute 7-Day Lat/Lon Track" to calculate.
        </div>
        """
  missing_str = ", ".join(missing)
  return f"""
    <div style="width:100%;box-sizing:border-box;padding:12px 16px;border-radius:10px;background:#1a1625;color:#fbbf24;border:1px solid #b45309;font-family:Inter,sans-serif;font-weight:600;">
      ⏳ {filled} / 5 TLE slots filled — Missing valid TLE for: {missing_str}
    </div>
    """


def compute_all_5(tle1, tle2, tle3, tle4, tle5, step_minutes):
  parsed, missing = validate_slots(tle1, tle2, tle3, tle4, tle5)
  if len(parsed) < 5:
    missing_str = ", ".join(missing)
    err_html = f"""
        <div style="width:100%;box-sizing:border-box;padding:18px;border-radius:12px;background:#161b26;color:#fbbf24;border:1px solid #b45309;font-family:Inter,sans-serif;">
          <b>Waiting for all 5 satellites ({len(parsed)}/5 completed).</b><br>
          Please paste valid TLE data under: <b>{missing_str}</b>.
        </div>
        """
    return err_html, pd.DataFrame()

  step_val = int(step_minutes) if step_minutes else 30
  ts = load.timescale()
  now_utc = datetime.now(timezone.utc)
  total_steps = int((7 * 24 * 60) / step_val) + 1
  times_dt = [now_utc + timedelta(minutes=k * step_val) for k in range(total_steps)]
  t_sf = ts.utc(
      [d.year for d in times_dt],
      [d.month for d in times_dt],
      [d.day for d in times_dt],
      [d.hour for d in times_dt],
      [d.minute for d in times_dt],
      [d.second for d in times_dt],
  )

  rows = []
  for label_name, l1, l2 in parsed:
    sat = EarthSatellite(l1, l2, label_name, ts)
    geoc = sat.at(t_sf)
    sub = geoc.subpoint()
    for dt_item, lat, lon, alt in zip(
        times_dt,
        sub.latitude.degrees,
        sub.longitude.degrees,
        sub.elevation.km,
    ):
      rows.append({
          "Slot / Satellite": label_name,
          "UTC Time": dt_item.strftime("%Y-%m-%d %H:%M"),
          "Latitude (deg)": round(float(lat), 3),
          "Longitude (deg)": round(float(lon), 3),
          "Altitude (km)": round(float(alt), 1),
      })

  df = pd.DataFrame(rows)
  min_lat, max_lat = df["Latitude (deg)"].min(), df["Latitude (deg)"].max()
  min_lon, max_lon = df["Longitude (deg)"].min(), df["Longitude (deg)"].max()

  summary_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">ALL 5 SATELLITES COMPUTED • NEXT 7 DAYS (UTC)</div>
        <div style="font-size:19px;font-weight:700;color:#f8fafc;margin-top:2px;">Intelsat • Eutelsat • Starlink • Qianfan • Kuiper <span style="font-size:13px;color:#94a3b8;">({len(df)} rows)</span></div>
      </div>

      <div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">LATITUDE RANGE (7D)</div>
          <div style="font-size:16px;font-weight:700;color:#38bdf8;margin-top:4px;">{min_lat:.2f}° to {max_lat:.2f}°</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">LONGITUDE RANGE (7D)</div>
          <div style="font-size:16px;font-weight:700;color:#a78bfa;margin-top:4px;">{min_lon:.2f}° to {max_lon:.2f}°</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">TIME STEP</div>
          <div style="font-size:16px;font-weight:700;color:#34d399;margin-top:4px;">Every {step_val} min (168h)</div>
        </div>
      </div>
    </div>
    """
  return summary_html, df


full_width_css = """
html, body, .gradio-container, .gradio-container > .main, .gradio-container > .main > .wrap, .block, .form {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}
.gradio-container {
  padding: 10px 14px !important;
}
"""

with gr.Blocks() as demo:
  gr.Markdown(
      "## 🌍 5-Satellite 7-Day Latitude & Longitude Ground-Track Calculator"
  )

  with gr.Row(equal_height=False):
    with gr.Column(scale=5, min_width=340):
      status_box = gr.HTML(value=update_status_badge("", "", "", "", ""))

      sel_1 = gr.Dropdown(
          choices=["Intelsat"], value="Intelsat", label="Slot 1 Satellite"
      )
      tle_1 = gr.Textbox(
          lines=3,
          label="Intelsat TLE",
          placeholder="Paste Intelsat TLE (name + Line 1 + Line 2)...",
      )

      sel_2 = gr.Dropdown(
          choices=["Eutelsat"], value="Eutelsat", label="Slot 2 Satellite"
      )
      tle_2 = gr.Textbox(
          lines=3,
          label="Eutelsat TLE",
          placeholder="Paste Eutelsat TLE (name + Line 1 + Line 2)...",
      )

      sel_3 = gr.Dropdown(
          choices=["Starlink"], value="Starlink", label="Slot 3 Satellite"
      )
      tle_3 = gr.Textbox(
          lines=3,
          label="Starlink TLE",
          placeholder="Paste Starlink TLE (name + Line 1 + Line 2)...",
      )

      sel_4 = gr.Dropdown(
          choices=["Qianfan"], value="Qianfan", label="Slot 4 Satellite"
      )
      tle_4 = gr.Textbox(
          lines=3,
          label="Qianfan TLE",
          placeholder="Paste Qianfan TLE (name + Line 1 + Line 2)...",
      )

      sel_5 = gr.Dropdown(
          choices=["Kuiper"], value="Kuiper", label="Slot 5 Satellite"
      )
      tle_5 = gr.Textbox(
          lines=3,
          label="Kuiper TLE",
          placeholder="Paste Kuiper TLE (name + Line 1 + Line 2)...",
      )

      step_dd = gr.Dropdown(
          choices=["15", "30", "60", "120"],
          value="30",
          label="Sampling Step (Minutes)",
      )
      calc_btn = gr.Button(
          "⚡ Compute 7-Day Lat/Lon Track (All 5)", variant="primary"
      )

    with gr.Column(scale=7, min_width=440):
      out_summary = gr.HTML()
      out_table = gr.Dataframe(
          label="7-Day Latitude / Longitude Schedule (UTC)", interactive=False
      )

  slot_inputs = [tle_1, tle_2, tle_3, tle_4, tle_5]
  for tb in slot_inputs:
    tb.change(fn=update_status_badge, inputs=slot_inputs, outputs=[status_box])

  calc_btn.click(
      fn=compute_all_5,
      inputs=[tle_1, tle_2, tle_3, tle_4, tle_5, step_dd],
      outputs=[out_summary, out_table],
  )

demo.launch(share=False, theme=gr.themes.Soft(), css=full_width_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>